# Monthly Credit Monitoring

Update `REPORT_MONTH` in the Parameters cell below, then **Run All**.

In [1]:
import snowflake.connector
import tomli as tomllib
import pandas as pd
import datetime
from pathlib import Path

cfg = tomllib.loads(Path.home().joinpath(".snowflake/connections.toml").read_text())
profile = cfg["A6040307054171-BANK_NOVO_ENTERPRISE"]

conn = snowflake.connector.connect(
    account=profile["account"],
    user=profile["user"],
    authenticator=profile.get("authenticator"),
    role="BI_ROLE",
    warehouse="COMPUTE_WH",
    database="PROD_DB",
    schema="ADHOC",
)
print("Connected")

/Users/markpanjaitan/Library/Python/3.9/lib/python/site-packages/snowflake/connector/vendored/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...


/Users/markpanjaitan/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Connected


## Parameters

In [2]:
REPORT_MONTH = "2026-06"  # ← Update this each month (YYYY-MM)
FF_EXCL_GROUP = "75fe98d2-6549-46a1-aa04-a1c621e21d9e"

y, m = int(REPORT_MONTH[:4]), int(REPORT_MONTH[5:7])
prev_dt = datetime.date(y, m, 1) - datetime.timedelta(days=1)
PREV_MONTH = prev_dt.strftime("%Y-%m")
next_m = m % 12 + 1
next_y = y + (1 if m == 12 else 0)
MONTH_END = str(datetime.date(next_y, next_m, 1) - datetime.timedelta(days=1))

print(f"Report month  : {REPORT_MONTH}")
print(f"Previous month: {PREV_MONTH}")
print(f"Month-end date: {MONTH_END}")

Report month  : 2026-06
Previous month: 2026-05
Month-end date: 2026-06-30


In [3]:
def sql(query: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(query)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

def label(n: int, title: str) -> None:
    print(f"\n{'─'*62}\n  [{n:02d}]  {title}\n{'─'*62}")

def copy(df: pd.DataFrame) -> None:
    """Copy a DataFrame to the clipboard as tab-separated values."""
    df.to_clipboard(index=False)
    print("✓ Copied to clipboard")

---
# Credit Card (CC) Metrics

## 1. Monthly Bookings — Volume, FICO, Credit Limit

Approved accounts per month with average FICO (from most recent invitation) and average credit limit, plus MoM % change and trailing 6-month weighted averages.

In [4]:
label(1, "CC-1 · Monthly Bookings — Volume, FICO, Credit Limit")
df_bookings = sql(f"""
WITH invitations AS (
    SELECT business_id, fico_score
    FROM (
        SELECT business_id, fico_score,
            ROW_NUMBER() OVER (PARTITION BY business_id ORDER BY created_at DESC) AS rn
        FROM fivetran_db.prod_novo_api_public.credit_card_invitations
    )
    WHERE rn = 1
),
base AS (
    SELECT
        TO_CHAR(a.created_at, 'YYYY-MM') AS year_month,
        COUNT(*)                                        AS total,
        ROUND(AVG(i.fico_score), 0)                    AS avg_fico,
        ROUND(AVG(a.credit_limit / 100), 0)            AS avg_credit_limit
    FROM fivetran_db.prod_novo_api_public.credit_card_applications a
    LEFT JOIN invitations i ON a.business_id = i.business_id
    WHERE a.business_id NOT IN (
        SELECT business_id
        FROM fivetran_db.prod_novo_api_public.business_group_assignments
        WHERE business_group_id = '{FF_EXCL_GROUP}'
    )
    AND a.created_at >= '2024-11-01'
    AND a.status = 'APPROVED'
    GROUP BY TO_CHAR(a.created_at, 'YYYY-MM')
),
numbered AS (
    SELECT
        year_month, total, avg_fico, avg_credit_limit,
        LAG(total)             OVER (ORDER BY year_month) AS prev_total,
        LAG(avg_fico)          OVER (ORDER BY year_month) AS prev_fico,
        LAG(avg_credit_limit)  OVER (ORDER BY year_month) AS prev_credit_limit,
        ROW_NUMBER()           OVER (ORDER BY year_month) AS rn
    FROM base
)
SELECT
    year_month,
    total,
    CASE WHEN prev_total IS NOT NULL AND prev_total != 0
        THEN ROUND((total - prev_total)::FLOAT / prev_total * 100, 2)
    END AS total_mom_pct_change,
    CASE WHEN rn >= 7
        THEN ROUND(AVG(total) OVER (ORDER BY year_month ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING), 2)
    END AS total_trailing_6mo_avg,
    avg_fico,
    CASE WHEN prev_fico IS NOT NULL AND prev_fico != 0
        THEN ROUND((avg_fico - prev_fico)::FLOAT / prev_fico * 100, 2)
    END AS fico_mom_pct_change,
    CASE WHEN rn >= 7
        THEN ROUND(
            SUM(avg_fico * total) OVER (ORDER BY year_month ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING)
            / NULLIF(SUM(total) OVER (ORDER BY year_month ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING), 0)
        , 0)
    END AS fico_weighted_trailing_6mo_avg,
    avg_credit_limit,
    CASE WHEN prev_credit_limit IS NOT NULL AND prev_credit_limit != 0
        THEN ROUND((avg_credit_limit - prev_credit_limit)::FLOAT / prev_credit_limit * 100, 2)
    END AS credit_limit_mom_pct_change,
    CASE WHEN rn >= 7
        THEN ROUND(
            SUM(avg_credit_limit * total) OVER (ORDER BY year_month ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING)
            / NULLIF(SUM(total) OVER (ORDER BY year_month ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING), 0)
        , 0)
    END AS credit_limit_weighted_trailing_6mo_avg
FROM numbered
ORDER BY year_month
""")
df_bookings


──────────────────────────────────────────────────────────────
  [01]  CC-1 · Monthly Bookings — Volume, FICO, Credit Limit
──────────────────────────────────────────────────────────────


,YEAR_MONTH,TOTAL,TOTAL_MOM_PCT_CHANGE,TOTAL_TRAILING_6MO_AVG,AVG_FICO,FICO_MOM_PCT_CHANGE,FICO_WEIGHTED_TRAILING_6MO_AVG,AVG_CREDIT_LIMIT,CREDIT_LIMIT_MOM_PCT_CHANGE,CREDIT_LIMIT_WEIGHTED_TRAILING_6MO_AVG
0,2024-11,92,NaN,None,694,NaN,NaN,5810,NaN,NaN
1,2024-12,482,423.91,None,696,0.29,NaN,5947,2.36,NaN
2,2025-01,1002,107.88,None,690,-0.86,NaN,5690,-4.32,NaN
3,2025-02,475,-52.59,None,704,2.03,NaN,6242,9.70,NaN
4,2025-03,512,7.79,None,684,-2.84,NaN,5439,-12.86,NaN
5,2025-04,568,10.94,None,689,0.73,NaN,5606,3.07,NaN
6,2025-05,105,-81.51,521.83,704,2.18,692.0,6410,14.34,5761.0
7,2025-06,508,383.81,524.00,700,-0.57,692.0,6319,-1.42,5781.0
8,2025-07,503,-0.98,528.33,687,-1.86,693.0,5352,-15.30,5842.0
9,2025-08,145,-71.17,445.17,687,0.00,693.0,5586,4.37,5806.0


## 2. Invite-to-Book Conversion Within 30 Days (by Invitation Month)

For each month invitations were sent, how many distinct businesses booked (approved application) within 30 days of their invitation. Note: the most recent invitation month's window may not be fully closed yet.

In [5]:
label(2, "CC-2 · Invite-to-Book Conversion (30 days)")
df_invite_conversion = sql(f"""
SELECT
    TO_CHAR(i.created_at, 'YYYY-MM')                           AS invite_month,
    COUNT(DISTINCT i.business_id)                              AS total_invited,
    COUNT(DISTINCT CASE
        WHEN a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
        THEN i.business_id
    END)                                                       AS booked_within_30d,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
            THEN i.business_id
        END)
        / NULLIF(COUNT(DISTINCT i.business_id), 0) * 100
    , 2)                                                       AS conversion_rate_pct
FROM fivetran_db.prod_novo_api_public.credit_card_invitations i
LEFT JOIN fivetran_db.prod_novo_api_public.credit_card_applications a
    ON  i.business_id = a.business_id
    AND a.status = 'APPROVED'
    AND a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
WHERE i.created_at >= '2024-11-01'
  AND i.business_id NOT IN (
      SELECT business_id
      FROM fivetran_db.prod_novo_api_public.business_group_assignments
      WHERE business_group_id = '{FF_EXCL_GROUP}'
  )
GROUP BY 1
ORDER BY 1
""")
df_invite_conversion


──────────────────────────────────────────────────────────────
  [02]  CC-2 · Invite-to-Book Conversion (30 days)
──────────────────────────────────────────────────────────────


,INVITE_MONTH,TOTAL_INVITED,BOOKED_WITHIN_30D,CONVERSION_RATE_PCT
0,2024-11,2511,137,5.46
1,2024-12,9293,551,5.93
2,2025-01,20670,1363,6.59
3,2025-03,13433,739,5.50
4,2025-04,11030,429,3.89
5,2025-06,23471,775,3.30
6,2025-07,8562,381,4.45
7,2025-11,22363,729,3.26
8,2026-01,23572,598,2.54
9,2026-03,24795,789,3.18


## 3. Monthly Bookings by Risk Bucket

Approved accounts per booking month broken down by risk tier (1–5 or Reject Inference). Risk bucket assigned using the same logic as `MONITOR_RISK_BUCKET_LOOKUP`: last-invite month determines which score table to use, with COALESCE fallbacks for April and May 2026.

In [6]:
label(3, "CC-3 · Monthly Bookings by Risk Bucket")
df_risk_raw = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
approved_apps AS (
    SELECT business_id, TO_CHAR(created_at, 'YYYY-MM') AS booking_month
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS
    WHERE status = 'APPROVED'
      AND created_at >= '2024-11-01'
      AND business_id NOT IN (
          SELECT business_id
          FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
)
SELECT
    a.booking_month,
    COALESCE(
        CASE
            WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-04'
                THEN COALESCE(apr.final_risk_bucket::TEXT, retro.final_risk_bucket::TEXT)
            WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-05'
                THEN COALESCE(may."final_risk_bucket"::TEXT, apr.final_risk_bucket::TEXT)
            ELSE retro.final_risk_bucket::TEXT
        END,
        'Reject Inference'
    ) AS risk_bucket,
    COUNT(*) AS accounts
FROM approved_apps a
LEFT JOIN latest_invite li                             ON li.business_id   = a.business_id
LEFT JOIN PROD_DB.ADHOC.RISK_BUCKET_RETRO_SCORE_MAY4  retro ON retro.business_id = a.business_id
LEFT JOIN PROD_DB.ADHOC.CC_APR_CAMPAIGN_MODEL_BUCKET  apr   ON apr.business_id   = a.business_id
LEFT JOIN PROD_DB.DE.CC_NEW_ACCOUNT_MODEL_UW           may   ON may.business_id   = a.business_id
GROUP BY 1, 2
ORDER BY 1, 2
""")

# Pivot so each risk bucket is a column
df_risk_buckets = df_risk_raw.pivot_table(
    index='BOOKING_MONTH', columns='RISK_BUCKET', values='ACCOUNTS', fill_value=0
).reset_index()
df_risk_buckets.columns.name = None

# Put Reject Inference last
cols = ['BOOKING_MONTH'] + [c for c in df_risk_buckets.columns if c not in ('BOOKING_MONTH', 'Reject Inference')]
if 'Reject Inference' in df_risk_buckets.columns:
    cols.append('Reject Inference')
df_risk_buckets[cols]


──────────────────────────────────────────────────────────────
  [03]  CC-3 · Monthly Bookings by Risk Bucket
──────────────────────────────────────────────────────────────


,BOOKING_MONTH,1,2,3,4,5,Reject Inference
0,2024-11,25.0,13.0,16.0,17.0,21.0,0.0
1,2024-12,133.0,86.0,89.0,93.0,81.0,0.0
2,2025-01,210.0,191.0,223.0,194.0,184.0,0.0
3,2025-02,128.0,95.0,116.0,78.0,58.0,0.0
4,2025-03,110.0,79.0,94.0,119.0,110.0,0.0
5,2025-04,112.0,114.0,98.0,114.0,130.0,0.0
6,2025-05,30.0,27.0,19.0,12.0,17.0,0.0
7,2025-06,120.0,109.0,97.0,93.0,89.0,0.0
8,2025-07,111.0,76.0,87.0,117.0,112.0,0.0
9,2025-08,25.0,36.0,30.0,34.0,20.0,0.0


## 4. Monthly Bookings by Decisioning Model

Approved accounts per booking month broken down by the underwriting campaign active at the time of their last invitation:

| Model | Last invite window |
|---|---|
| Foundational Testing | 2024-11 → 2025-07 |
| CART | 2025-11 → 2026-01 |
| New Account Model | 2026-03+ |
| Other | Gap months (2025-08–10, 2026-02) — may still have retro scores; not the same as Reject Inference |

In [7]:
label(4, "CC-4 · Monthly Bookings by Decisioning Model")
df_model_raw = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
approved_apps AS (
    SELECT business_id, TO_CHAR(created_at, 'YYYY-MM') AS booking_month
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS
    WHERE status = 'APPROVED'
      AND created_at >= '2024-11-01'
      AND business_id NOT IN (
          SELECT business_id
          FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
)
SELECT
    a.booking_month,
    CASE
        WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') BETWEEN '2024-11' AND '2025-07' THEN 'Foundational Testing'
        WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') BETWEEN '2025-11' AND '2026-01' THEN 'CART'
        WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') >= '2026-03'                    THEN 'New Account Model'
        ELSE 'Other'
    END AS decisioning_model,
    COUNT(*) AS accounts
FROM approved_apps a
LEFT JOIN latest_invite li ON li.business_id = a.business_id
GROUP BY 1, 2
ORDER BY 1, 2
""")

model_order = ['Foundational Testing', 'CART', 'New Account Model', 'Other']
df_model = df_model_raw.pivot_table(
    index='BOOKING_MONTH', columns='DECISIONING_MODEL', values='ACCOUNTS', fill_value=0
).reset_index()
df_model.columns.name = None
cols = ['BOOKING_MONTH'] + [c for c in model_order if c in df_model.columns]
df_model[cols]


──────────────────────────────────────────────────────────────
  [04]  CC-4 · Monthly Bookings by Decisioning Model
──────────────────────────────────────────────────────────────


,BOOKING_MONTH,Foundational Testing,CART,New Account Model
0,2024-11,92.0,0.0,0.0
1,2024-12,482.0,0.0,0.0
2,2025-01,1002.0,0.0,0.0
3,2025-02,475.0,0.0,0.0
4,2025-03,512.0,0.0,0.0
5,2025-04,568.0,0.0,0.0
6,2025-05,105.0,0.0,0.0
7,2025-06,508.0,0.0,0.0
8,2025-07,503.0,0.0,0.0
9,2025-08,145.0,0.0,0.0


## 5. 30-Day Invite-to-Book Conversion Rate by Risk Bucket × Invitation Month

For each invitation month and risk bucket: % of invited businesses that booked within 30 days. Risk bucket is assigned using the same last-invite logic as CC #3.

In [8]:
label(5, "CC-5 · Conversion Rate by Risk Bucket × Invite Month")
df_conv_bucket_raw = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
risk_lookup AS (
    SELECT
        li.business_id,
        COALESCE(
            CASE
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-04'
                    THEN COALESCE(apr.final_risk_bucket::TEXT, retro.final_risk_bucket::TEXT)
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-05'
                    THEN COALESCE(may."final_risk_bucket"::TEXT, apr.final_risk_bucket::TEXT)
                ELSE retro.final_risk_bucket::TEXT
            END,
            'Reject Inference'
        ) AS risk_bucket
    FROM latest_invite li
    LEFT JOIN PROD_DB.ADHOC.RISK_BUCKET_RETRO_SCORE_MAY4 retro ON retro.business_id = li.business_id
    LEFT JOIN PROD_DB.ADHOC.CC_APR_CAMPAIGN_MODEL_BUCKET apr   ON apr.business_id   = li.business_id
    LEFT JOIN PROD_DB.DE.CC_NEW_ACCOUNT_MODEL_UW           may   ON may.business_id   = li.business_id
)
SELECT
    TO_CHAR(i.created_at, 'YYYY-MM')                           AS invite_month,
    rl.risk_bucket,
    COUNT(DISTINCT i.business_id)                              AS total_invited,
    COUNT(DISTINCT CASE
        WHEN a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
        THEN i.business_id
    END)                                                       AS booked_within_30d,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
            THEN i.business_id
        END) / NULLIF(COUNT(DISTINCT i.business_id), 0) * 100
    , 2)                                                       AS conversion_rate_pct
FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS i
LEFT JOIN risk_lookup rl ON rl.business_id = i.business_id
LEFT JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS a
    ON  i.business_id = a.business_id
    AND a.status = 'APPROVED'
    AND a.created_at BETWEEN i.created_at AND DATEADD(day, 30, i.created_at)
WHERE i.created_at >= '2024-11-01'
  AND i.business_id NOT IN (
      SELECT business_id
      FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
      WHERE business_group_id = '{FF_EXCL_GROUP}'
  )
GROUP BY 1, 2
ORDER BY 1, 2
""")

# Pivot: rows = invite_month, columns = risk bucket, values = conversion_rate_pct
df_conv_bucket = df_conv_bucket_raw.pivot_table(
    index='INVITE_MONTH', columns='RISK_BUCKET', values='CONVERSION_RATE_PCT', fill_value=0
).reset_index()
df_conv_bucket.columns.name = None

# Order columns: buckets 1-5 then Reject Inference
bucket_cols = sorted([c for c in df_conv_bucket.columns if c not in ('INVITE_MONTH', 'Reject Inference')])
if 'Reject Inference' in df_conv_bucket.columns:
    bucket_cols.append('Reject Inference')
df_conv_bucket[['INVITE_MONTH'] + bucket_cols]


──────────────────────────────────────────────────────────────
  [05]  CC-5 · Conversion Rate by Risk Bucket × Invite Month
──────────────────────────────────────────────────────────────


/var/folders/v3/m1_xbhs97_j26szjgrmgyvk40000gn/T/ipykernel_61738/2881448147.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_conv_bucket = df_conv_bucket_raw.pivot_table(


,INVITE_MONTH,1,2,3,4,5,Reject Inference
0,2024-11,4.65,7.36,25.00,47.92,89.66,0.00
1,2024-12,3.84,7.28,22.17,40.37,85.45,0.00
2,2025-01,3.55,8.35,27.80,40.52,89.60,0.00
3,2025-03,3.45,7.98,22.72,40.59,87.65,0.00
4,2025-04,1.83,4.45,14.17,28.67,91.06,0.00
5,2025-06,1.88,4.09,12.15,24.01,86.11,0.00
6,2025-07,1.95,4.46,10.70,25.30,85.84,0.00
7,2025-11,1.83,2.62,8.89,17.47,90.67,0.00
8,2026-01,1.64,2.77,7.07,10.90,100.00,0.00
9,2026-03,2.25,2.84,5.32,13.06,100.00,4.28


## 6. Invite-to-Book Conversion Rate by Booking Cohort

For each booking cohort: % of accounts that booked within 30 days of their last invitation, plus total bookings, raw 30-day converter count, and MoM change in conversion rate. Unlike CC-2 (which groups by invite month), this groups by the month of approval.

In [ ]:
label(6, "CC-6 · Invite-to-Book Conversion Rate by Booking Cohort")
df_cohort_conversion = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
base AS (
    SELECT
        TO_CHAR(a.created_at, 'YYYY-MM')                          AS booking_month,
        COUNT(*)                                                    AS total_booked,
        COUNT(CASE
            WHEN a.created_at BETWEEN li.last_invite_at
                             AND DATEADD(day, 30, li.last_invite_at)
            THEN 1
        END)                                                        AS booked_within_30d,
        ROUND(
            COUNT(CASE
                WHEN a.created_at BETWEEN li.last_invite_at
                                 AND DATEADD(day, 30, li.last_invite_at)
                THEN 1
            END) / NULLIF(COUNT(*), 0) * 100
        , 2)                                                        AS conversion_rate_pct
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS a
    LEFT JOIN latest_invite li ON li.business_id = a.business_id
    WHERE a.status = 'APPROVED'
      AND a.created_at >= '2024-11-01'
      AND a.business_id NOT IN (
          SELECT business_id
          FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
    GROUP BY 1
)
SELECT
    booking_month,
    total_booked,
    booked_within_30d,
    conversion_rate_pct,
    CASE
        WHEN LAG(conversion_rate_pct) OVER (ORDER BY booking_month) IS NOT NULL
             AND LAG(conversion_rate_pct) OVER (ORDER BY booking_month) != 0
        THEN ROUND(
            (conversion_rate_pct - LAG(conversion_rate_pct) OVER (ORDER BY booking_month))
            / LAG(conversion_rate_pct) OVER (ORDER BY booking_month) * 100
        , 2)
    END AS conversion_rate_mom_pct_change
FROM base
ORDER BY booking_month
""")
df_cohort_conversion

---
## Spend & Balances

Vintage curves by booking month. Rows = booking cohort, columns = statement number since booking. Excludes the current booking month (incomplete cohort) and any statement number that hasn't fully closed yet for that vintage.

In [9]:
label(7, "CC-7 · Vintage Raw (preview)")
df_vintage_raw = sql(f"""
WITH loan_tape_updated AS (
    SELECT
        a.*,
        b.business_id,
        ROW_NUMBER() OVER (PARTITION BY b.business_id, a.statement_date ORDER BY a.record_version DESC) AS rn
    FROM PROD_DB.DATA.CREDIT_CARD_ACCOUNT_LOAN_TAPE_HISTORY a
    JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_ACCOUNTS b
        ON a.account_id = b.external_account_id
    WHERE b.business_id NOT IN (
        SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
        WHERE business_group_id = '{FF_EXCL_GROUP}'
    )
    AND a.billing_period_number >= 1
),
booking_months AS (
    SELECT business_id, TO_CHAR(MIN(created_at), 'YYYY-MM') AS booking_month,
        MIN(created_at)::DATE AS booking_date
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS
    WHERE status = 'APPROVED'
      AND business_id NOT IN (
          SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
    GROUP BY 1
)
SELECT
    bm.booking_month,
    CASE WHEN lt.billing_period_number - 1 = 0
         THEN lt.billing_period_number
         ELSE lt.billing_period_number - 1
    END                                                                             AS stmt_no,
    COUNT(DISTINCT lt.business_id)                                                  AS open_accounts,
    ROUND(SUM(s.purchases) / 100.0
        / NULLIF(COUNT(DISTINCT lt.business_id), 0), 2)                            AS avg_spend_per_account,
    ROUND(SUM(CASE WHEN lt.days_past_due <= 210 THEN lt.ending_balance END) / 100.0
        / NULLIF(COUNT(DISTINCT CASE WHEN lt.days_past_due <= 210
                                     THEN lt.business_id END), 0), 2)              AS avg_balance_per_account
FROM (SELECT * FROM loan_tape_updated WHERE rn = 1) lt
JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_STATEMENTS s
    ON  lt.business_id = s.business_id
    AND TO_CHAR(lt.statement_date, 'YYYY-MM-DD') = TO_CHAR(s.created_at, 'YYYY-MM-DD')
JOIN booking_months bm ON lt.business_id = bm.business_id
WHERE lt.billing_period_number != 0
  AND bm.booking_month < TO_CHAR(DATEADD(month, -1, CURRENT_DATE), 'YYYY-MM')
  AND CASE WHEN lt.billing_period_number - 1 = 0
           THEN lt.billing_period_number
           ELSE lt.billing_period_number - 1
      END < DATEDIFF(month, bm.booking_date, CURRENT_DATE)
GROUP BY 1, 2
ORDER BY 1, 2
""")
df_vintage_raw.head(10)


──────────────────────────────────────────────────────────────
  [06]  CC-6 · Vintage Raw (preview)
──────────────────────────────────────────────────────────────


,BOOKING_MONTH,STMT_NO,OPEN_ACCOUNTS,AVG_SPEND_PER_ACCOUNT,AVG_BALANCE_PER_ACCOUNT
0,2024-11,1,92,685.32,581.24
1,2024-11,2,92,635.16,743.39
2,2024-11,3,92,536.50,848.71
3,2024-11,4,92,463.95,995.93
4,2024-11,5,92,558.83,1295.78
5,2024-11,6,92,425.73,1331.52
6,2024-11,7,92,475.03,1358.34
7,2024-11,8,92,344.01,1371.54
8,2024-11,9,92,422.50,1401.66
9,2024-11,10,92,286.01,1226.79


### Average Spend per Account by Statement Number × Booking Vintage

In [10]:
label(8, "CC-8 · Avg Spend per Account by Statement × Vintage")
df_spend = df_vintage_raw.pivot_table(
    index='BOOKING_MONTH', columns='STMT_NO', values='AVG_SPEND_PER_ACCOUNT'
).round(2)
df_spend.columns = [f'Stmt {int(c)}' for c in df_spend.columns]
df_spend.columns.name = None
df_spend


──────────────────────────────────────────────────────────────
  [07]  CC-7 · Avg Spend per Account by Statement × Vintage
──────────────────────────────────────────────────────────────


,Stmt 1,Stmt 2,Stmt 3,Stmt 4,Stmt 5,Stmt 6,Stmt 7,Stmt 8,Stmt 9,Stmt 10,Stmt 11,Stmt 12,Stmt 13,Stmt 14,Stmt 15,Stmt 16,Stmt 17,Stmt 18
BOOKING_MONTH,,,,,,,,,,,,,,,,,,
2024-11,685.32,635.16,536.5,463.95,558.83,425.73,475.03,344.01,422.5,286.01,444.16,287.7,382.08,360.45,302.95,404.61,438.71,395.1
2024-12,540.45,688.58,593.02,670.61,636.3,567.3,562.93,530.0,486.93,494.89,476.12,507.78,454.75,439.65,447.76,550.93,504.99,NaN
2025-01,603.33,797.68,694.22,603.83,626.73,531.31,579.31,562.75,501.82,546.52,490.24,454.6,526.95,472.7,625.34,570.6,NaN,NaN
2025-02,471.52,695.5,631.72,613.77,583.26,591.01,507.32,562.05,484.69,505.33,583.04,504.66,537.07,670.14,602.01,NaN,NaN,NaN
2025-03,645.13,561.87,653.25,509.67,525.13,520.82,467.8,450.5,418.52,429.03,365.11,340.77,465.12,496.71,NaN,NaN,NaN,NaN
2025-04,770.59,868.53,661.43,608.32,590.37,539.61,491.79,467.78,501.64,504.24,425.75,550.1,556.49,NaN,NaN,NaN,NaN,NaN
2025-05,387.14,591.92,825.16,683.99,770.87,489.81,594.04,676.22,777.74,801.18,480.62,489.19,NaN,NaN,NaN,NaN,NaN,NaN
2025-06,630.7,931.17,911.1,783.85,794.04,678.5,610.63,754.15,780.15,869.58,955.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07,570.88,690.62,590.52,626.56,506.6,538.46,463.73,489.85,555.83,536.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Average Balance Outstanding per Account by Statement Number × Booking Vintage

In [11]:
label(9, "CC-9 · Avg Balance per Account by Statement × Vintage")
df_balance = df_vintage_raw.pivot_table(
    index='BOOKING_MONTH', columns='STMT_NO', values='AVG_BALANCE_PER_ACCOUNT'
).round(2)
df_balance.columns = [f'Stmt {int(c)}' for c in df_balance.columns]
df_balance.columns.name = None
df_balance


──────────────────────────────────────────────────────────────
  [08]  CC-8 · Avg Balance per Account by Statement × Vintage
──────────────────────────────────────────────────────────────


,Stmt 1,Stmt 2,Stmt 3,Stmt 4,Stmt 5,Stmt 6,Stmt 7,Stmt 8,Stmt 9,Stmt 10,Stmt 11,Stmt 12,Stmt 13,Stmt 14,Stmt 15,Stmt 16,Stmt 17,Stmt 18
BOOKING_MONTH,,,,,,,,,,,,,,,,,,
2024-11,581.24,743.39,848.71,995.93,1295.78,1331.52,1358.34,1371.54,1401.66,1226.79,1235.82,1191.54,1362.14,1399.47,1337.4,1352.49,1406.8,1439.77
2024-12,440.33,744.54,900.47,1035.09,1150.08,1199.41,1322.46,1319.48,1411.57,1411.09,1442.73,1531.77,1541.31,1536.99,1623.24,1631.19,1610.06,NaN
2025-01,477.37,790.33,981.62,1037.73,1142.76,1227.67,1286.61,1373.74,1410.99,1445.42,1460.08,1442.24,1462.28,1544.55,1564.52,1632.88,NaN,NaN
2025-02,390.64,704.67,862.97,1013.81,1109.54,1217.43,1240.59,1323.16,1373.55,1375.81,1496.86,1518.27,1611.79,1666.67,1664.91,NaN,NaN,NaN
2025-03,505.34,790.38,949.5,1005.79,1128.23,1171.27,1227.0,1295.85,1351.28,1375.95,1376.75,1442.11,1426.23,1421.35,NaN,NaN,NaN,NaN
2025-04,543.97,873.27,1016.04,1198.97,1344.85,1454.21,1563.27,1651.78,1769.66,1835.5,1905.53,1983.2,2057.58,NaN,NaN,NaN,NaN,NaN
2025-05,365.06,706.28,790.35,970.27,930.81,935.95,1154.35,1129.9,1156.88,1402.28,1155.05,1223.47,NaN,NaN,NaN,NaN,NaN,NaN
2025-06,493.84,787.05,999.28,1074.6,1203.22,1336.23,1355.58,1484.98,1613.91,1748.55,1792.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07,453.06,790.4,967.71,1086.1,1155.72,1210.64,1313.14,1428.97,1500.75,1462.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Average Spend & Balance by Risk Bucket

Averaged across all booking vintages and statement periods. Values represent the average per account-statement observation within each bucket.

In [12]:
label(10, "CC-10 · Avg Spend & Balance by Risk Bucket")
df_by_bucket = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
risk_lookup AS (
    SELECT
        li.business_id,
        COALESCE(
            CASE
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-04'
                    THEN COALESCE(apr.final_risk_bucket::TEXT, retro.final_risk_bucket::TEXT)
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-05'
                    THEN COALESCE(may."final_risk_bucket"::TEXT, apr.final_risk_bucket::TEXT)
                ELSE retro.final_risk_bucket::TEXT
            END,
            'Reject Inference'
        ) AS risk_bucket
    FROM latest_invite li
    LEFT JOIN PROD_DB.ADHOC.RISK_BUCKET_RETRO_SCORE_MAY4 retro ON retro.business_id = li.business_id
    LEFT JOIN PROD_DB.ADHOC.CC_APR_CAMPAIGN_MODEL_BUCKET  apr   ON apr.business_id   = li.business_id
    LEFT JOIN PROD_DB.DE.CC_NEW_ACCOUNT_MODEL_UW           may   ON may.business_id   = li.business_id
),
loan_tape_updated AS (
    SELECT a.*, b.business_id,
        ROW_NUMBER() OVER (PARTITION BY b.business_id, a.statement_date ORDER BY a.record_version DESC) AS rn
    FROM PROD_DB.DATA.CREDIT_CARD_ACCOUNT_LOAN_TAPE_HISTORY a
    JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_ACCOUNTS b
        ON a.account_id = b.external_account_id
    WHERE b.business_id NOT IN (
        SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
        WHERE business_group_id = '{FF_EXCL_GROUP}'
    )
    AND a.billing_period_number >= 1
)
SELECT
    rl.risk_bucket,
    COUNT(*)                                                                        AS account_stmt_obs,
    COUNT(DISTINCT lt.business_id)                                                  AS distinct_accounts,
    ROUND(SUM(s.purchases) / 100.0 / NULLIF(COUNT(*), 0), 2)                      AS avg_spend_per_stmt,
    ROUND(SUM(CASE WHEN lt.days_past_due <= 210 THEN lt.ending_balance END) / 100.0
        / NULLIF(COUNT(CASE WHEN lt.days_past_due <= 210 THEN 1 END), 0), 2)       AS avg_balance_per_stmt
FROM (SELECT * FROM loan_tape_updated WHERE rn = 1) lt
JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_STATEMENTS s
    ON  lt.business_id = s.business_id
    AND TO_CHAR(lt.statement_date, 'YYYY-MM-DD') = TO_CHAR(s.created_at, 'YYYY-MM-DD')
JOIN risk_lookup rl ON rl.business_id = lt.business_id
WHERE lt.billing_period_number != 0
GROUP BY 1
ORDER BY 1
""")
df_by_bucket


──────────────────────────────────────────────────────────────
  [09]  CC-9 · Avg Spend & Balance by Risk Bucket
──────────────────────────────────────────────────────────────


,RISK_BUCKET,ACCOUNT_STMT_OBS,DISTINCT_ACCOUNTS,AVG_SPEND_PER_STMT,AVG_BALANCE_PER_STMT
0,1,16699,2072,772.09,828.64
1,2,12868,1380,633.00,1343.92
2,3,13626,1377,583.36,1423.70
3,4,13479,1411,483.73,1217.65
4,5,11815,992,368.59,1125.83
5,Reject Inference,324,324,418.47,303.83


### Average Spend & Balance by Booking Month × Risk Bucket

In [13]:
label(11, "CC-11 · Avg Spend & Balance by Booking Month × Risk Bucket")
df_month_bucket_raw = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
risk_lookup AS (
    SELECT
        li.business_id,
        COALESCE(
            CASE
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-04'
                    THEN COALESCE(apr.final_risk_bucket::TEXT, retro.final_risk_bucket::TEXT)
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-05'
                    THEN COALESCE(may."final_risk_bucket"::TEXT, apr.final_risk_bucket::TEXT)
                ELSE retro.final_risk_bucket::TEXT
            END,
            'Reject Inference'
        ) AS risk_bucket
    FROM latest_invite li
    LEFT JOIN PROD_DB.ADHOC.RISK_BUCKET_RETRO_SCORE_MAY4 retro ON retro.business_id = li.business_id
    LEFT JOIN PROD_DB.ADHOC.CC_APR_CAMPAIGN_MODEL_BUCKET  apr   ON apr.business_id   = li.business_id
    LEFT JOIN PROD_DB.DE.CC_NEW_ACCOUNT_MODEL_UW           may   ON may.business_id   = li.business_id
),
loan_tape_updated AS (
    SELECT a.*, b.business_id,
        ROW_NUMBER() OVER (PARTITION BY b.business_id, a.statement_date ORDER BY a.record_version DESC) AS rn
    FROM PROD_DB.DATA.CREDIT_CARD_ACCOUNT_LOAN_TAPE_HISTORY a
    JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_ACCOUNTS b
        ON a.account_id = b.external_account_id
    WHERE b.business_id NOT IN (
        SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
        WHERE business_group_id = '{FF_EXCL_GROUP}'
    )
    AND a.billing_period_number >= 1
),
booking_months AS (
    SELECT business_id, TO_CHAR(MIN(created_at), 'YYYY-MM') AS booking_month
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS
    WHERE status = 'APPROVED'
      AND business_id NOT IN (
          SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
    GROUP BY 1
)
SELECT
    bm.booking_month,
    rl.risk_bucket,
    COUNT(DISTINCT lt.business_id)                                                  AS distinct_accounts,
    ROUND(SUM(s.purchases) / 100.0 / NULLIF(COUNT(*), 0), 2)                      AS avg_spend_per_stmt,
    ROUND(SUM(CASE WHEN lt.days_past_due <= 210 THEN lt.ending_balance END) / 100.0
        / NULLIF(COUNT(CASE WHEN lt.days_past_due <= 210 THEN 1 END), 0), 2)       AS avg_balance_per_stmt
FROM (SELECT * FROM loan_tape_updated WHERE rn = 1) lt
JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_STATEMENTS s
    ON  lt.business_id = s.business_id
    AND TO_CHAR(lt.statement_date, 'YYYY-MM-DD') = TO_CHAR(s.created_at, 'YYYY-MM-DD')
JOIN risk_lookup rl   ON rl.business_id = lt.business_id
JOIN booking_months bm ON bm.business_id = lt.business_id
WHERE lt.billing_period_number != 0
GROUP BY 1, 2
ORDER BY 1, 2
""")

# Pivot: spend table (rows=booking_month, columns=risk_bucket)
print("--- Avg Spend per Statement ---")
df_spend_bucket = df_month_bucket_raw.pivot_table(
    index='BOOKING_MONTH', columns='RISK_BUCKET', values='AVG_SPEND_PER_STMT'
).round(2)
df_spend_bucket.columns.name = None
display(df_spend_bucket)

# Pivot: balance table
print("--- Avg Balance per Statement ---")
df_balance_bucket = df_month_bucket_raw.pivot_table(
    index='BOOKING_MONTH', columns='RISK_BUCKET', values='AVG_BALANCE_PER_STMT'
).round(2)
df_balance_bucket.columns.name = None
display(df_balance_bucket)


──────────────────────────────────────────────────────────────
  [10]  CC-10 · Avg Spend & Balance by Booking Month × Risk Bucket
──────────────────────────────────────────────────────────────
--- Avg Spend per Statement ---


,1,2,3,4,5,Reject Inference
BOOKING_MONTH,,,,,,
2024-11,868.0,369.01,347.59,192.53,219.41,NaN
2024-12,769.44,623.11,471.91,344.88,356.24,NaN
2025-01,721.93,644.63,678.16,488.03,285.72,NaN
2025-02,838.67,491.23,542.41,396.48,372.5,NaN
2025-03,517.34,611.74,377.8,641.65,299.13,NaN
2025-04,646.99,649.59,608.98,468.88,529.6,NaN
2025-05,1059.92,503.93,403.25,789.05,185.21,NaN
2025-06,1480.64,611.55,713.01,469.1,471.42,NaN
2025-07,681.09,680.99,609.15,503.91,365.97,NaN


--- Avg Balance per Statement ---


,1,2,3,4,5,Reject Inference
BOOKING_MONTH,,,,,,
2024-11,718.68,1817.05,1135.83,1200.9,1510.54,NaN
2024-12,975.4,1372.46,1487.59,1426.66,1261.98,NaN
2025-01,796.86,1491.92,1510.53,1349.67,1123.02,NaN
2025-02,1114.15,1184.2,1396.54,1368.8,1125.38,NaN
2025-03,694.89,1624.16,1299.04,1313.84,1034.9,NaN
2025-04,1050.31,1781.39,1929.42,1336.76,1350.98,NaN
2025-05,699.71,846.73,1510.69,1489.9,893.27,NaN
2025-06,1160.21,1324.28,1518.24,1059.3,1200.63,NaN
2025-07,702.78,1102.14,1614.69,1219.24,1097.36,NaN


---
## Economics

**NIBT = interchange + interest collected + fees collected − rewards accrued + charge-off amount**

Charge-off amount is already negative (loss). All figures normalized per origination account.

### Average Credit Limit by Booking Cohort × Risk Bucket

Approved credit line (in dollars) averaged across all accounts in each booking-month × risk-bucket cell. Risk bucket uses the same last-invite logic as the cells above.

In [14]:
label(12, "CC-12 · Avg Credit Limit by Cohort × Risk Bucket")
df_credit_line_raw = sql(f"""
WITH latest_invite AS (
    SELECT business_id, MAX(created_at) AS last_invite_at
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_INVITATIONS
    GROUP BY 1
),
risk_lookup AS (
    SELECT
        li.business_id,
        COALESCE(
            CASE
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-04'
                    THEN COALESCE(apr.final_risk_bucket::TEXT, retro.final_risk_bucket::TEXT)
                WHEN TO_CHAR(li.last_invite_at, 'YYYY-MM') = '2026-05'
                    THEN COALESCE(may."final_risk_bucket"::TEXT, apr.final_risk_bucket::TEXT)
                ELSE retro.final_risk_bucket::TEXT
            END,
            'Reject Inference'
        ) AS risk_bucket
    FROM latest_invite li
    LEFT JOIN PROD_DB.ADHOC.RISK_BUCKET_RETRO_SCORE_MAY4 retro ON retro.business_id = li.business_id
    LEFT JOIN PROD_DB.ADHOC.CC_APR_CAMPAIGN_MODEL_BUCKET  apr   ON apr.business_id   = li.business_id
    LEFT JOIN PROD_DB.DE.CC_NEW_ACCOUNT_MODEL_UW           may   ON may.business_id   = li.business_id
)
SELECT
    TO_CHAR(a.created_at, 'YYYY-MM')       AS booking_month,
    rl.risk_bucket,
    COUNT(*)                                AS accounts,
    ROUND(AVG(a.credit_limit / 100.0), 0)  AS avg_credit_limit
FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS a
LEFT JOIN risk_lookup rl ON rl.business_id = a.business_id
WHERE a.status = 'APPROVED'
  AND a.created_at >= '2024-11-01'
  AND a.business_id NOT IN (
      SELECT business_id
      FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
      WHERE business_group_id = '{FF_EXCL_GROUP}'
  )
GROUP BY 1, 2
ORDER BY 1, 2
""")

# Pivot: rows = booking_month, columns = risk_bucket, values = avg_credit_limit
df_credit_line = df_credit_line_raw.pivot_table(
    index='BOOKING_MONTH', columns='RISK_BUCKET', values='AVG_CREDIT_LIMIT'
).round(0)
df_credit_line.columns.name = None

# Columns ordered buckets 1–5 then Reject Inference
bucket_cols = sorted([c for c in df_credit_line.columns if c != 'Reject Inference'])
if 'Reject Inference' in df_credit_line.columns:
    bucket_cols.append('Reject Inference')
df_credit_line[bucket_cols]


──────────────────────────────────────────────────────────────
  [11]  CC-11 · Avg Credit Limit by Cohort × Risk Bucket
──────────────────────────────────────────────────────────────


,1,2,3,4,5,Reject Inference
BOOKING_MONTH,,,,,,
2024-11,11800.0,7885.0,2844.0,3029.0,1905.0,NaN
2024-12,12368.0,6198.0,3556.0,2591.0,1617.0,NaN
2025-01,12700.0,7204.0,4110.0,2441.0,1459.0,NaN
2025-02,11914.0,6716.0,4125.0,2917.0,1655.0,NaN
2025-03,12409.0,7316.0,4032.0,2634.0,1359.0,NaN
2025-04,12696.0,6912.0,4561.0,2649.0,1731.0,NaN
2025-05,12233.0,7019.0,3447.0,2417.0,1294.0,NaN
2025-06,13208.0,6399.0,4881.0,3269.0,1685.0,NaN
2025-07,12180.0,7013.0,3989.0,2457.0,1540.0,NaN


### Average Cumulative NIBT at Statement 3 by Risk Bucket

Only cohorts with at least 3 closed statement periods (stmt_no ≥ 3) are included.

In [15]:
label(13, "CC-13 · Cumulative NIBT at Statement 3 by Risk Bucket")
# Filter to stmt_no == 3 for cohorts that have reached at least 3 statements
at_stmt3 = df_nibt_raw[df_nibt_raw['STMT_NO'] == 3].copy()

# Average cumulative NIBT per origination at stmt 3, grouped by risk bucket
df_nibt_3mo = (
    at_stmt3
    .groupby('RISK_BUCKET', sort=False)
    .agg(
        cohorts_included=('BOOKING_MONTH', 'count'),
        avg_cum_nibt_per_orig=('CUM_NIBT_PER_ORIG', 'mean'),
    )
    .round(2)
    .reset_index()
    .sort_values('RISK_BUCKET')
)

# Also add an all-cohorts aggregate row
agg_row = pd.DataFrame([{
    'RISK_BUCKET': 'All Buckets',
    'cohorts_included': at_stmt3['BOOKING_MONTH'].nunique(),
    'avg_cum_nibt_per_orig': round(at_stmt3['CUM_NIBT_PER_ORIG'].mean(), 2),
}])
df_nibt_3mo = pd.concat([df_nibt_3mo, agg_row], ignore_index=True)
df_nibt_3mo


──────────────────────────────────────────────────────────────
  [12]  CC-12 · Cumulative NIBT at Statement 3 by Risk Bucket
──────────────────────────────────────────────────────────────


NameError: name 'df_nibt_raw' is not defined

---
## Risk Metrics

### 1. Min Pay Rates by Cohort

For each booking cohort: % of statements where the account paid in full, paid only the minimum (≥90% of min due but <90% of statement balance), or missed the minimum entirely. Denominator = statements with a non-zero minimum payment due that have passed their due date.

In [ ]:
label(14, "CC-14 · Min Pay Rates by Cohort")
df_min_pay = sql(f"""
WITH booking_months AS (
    SELECT business_id, TO_CHAR(MIN(created_at), 'YYYY-MM') AS booking_month
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_APPLICATIONS
    WHERE status = 'APPROVED'
      AND business_id NOT IN (
          SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
          WHERE business_group_id = '{FF_EXCL_GROUP}'
      )
    GROUP BY 1
),
stmt_payments AS (
    SELECT
        s.business_id,
        s.minimum_payment_due / 100.0   AS min_due,
        s.statement_balance   / 100.0   AS stmt_balance,
        COALESCE(SUM(CASE WHEN p.amount > 0 AND p.status = 'settled'
                          THEN p.amount / 100.0 END), 0) AS total_paid
    FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_STATEMENTS s
    LEFT JOIN FIVETRAN_DB.PROD_NOVO_API_PUBLIC.CREDIT_CARD_PAYMENTS p
        ON  s.business_id       = p.business_id
        AND p.created_at::DATE  > s.end_date::DATE
        AND p.created_at::DATE <= s.payment_due_date::DATE
    WHERE s.business_id NOT IN (
        SELECT business_id FROM FIVETRAN_DB.PROD_NOVO_API_PUBLIC.BUSINESS_GROUP_ASSIGNMENTS
        WHERE business_group_id = '{FF_EXCL_GROUP}'
    )
    AND s.minimum_payment_due > 0
    AND s.payment_due_date < CURRENT_DATE
    GROUP BY 1, 2, 3
)
SELECT
    bm.booking_month,
    COUNT(*)                                                                           AS statement_obs,
    ROUND(100.0 * SUM(CASE WHEN sp.total_paid >= sp.stmt_balance * 0.9
                           THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1)               AS paid_in_full_pct,
    ROUND(100.0 * SUM(CASE WHEN sp.total_paid >= sp.min_due * 0.9
                                AND sp.total_paid < sp.stmt_balance * 0.9
                           THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1)               AS min_pay_only_pct,
    ROUND(100.0 * SUM(CASE WHEN sp.total_paid < sp.min_due * 0.9
                           THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1)               AS missed_min_pay_pct
FROM stmt_payments sp
JOIN booking_months bm ON bm.business_id = sp.business_id
GROUP BY 1
ORDER BY 1
""")
df_min_pay

### 2. Missed First Payment Rate by Cohort

For accounts that have reached statement 2: DPD ≥ 1 at statement 2 means the statement-1 payment was not made by its due date.

---
# MCA Metrics